# One shared image through four saved melanoma pipelines

**Inference only. No training and no new threshold selection. Google Colab: T4 GPU.**

The selected image is **ISIC_0000056.jpg**, labelled melanoma in the official ISIC 2016
test labels. It was chosen before checking predictions: the first melanoma by filename
in the intersection of the official test set and the TransUNet held-out split recreated
from the supplied notebook. The split reconstruction matches its 256 test images and
50 melanoma images. ResUNet was developed on a 300-case subset of the official training
set, so this official test image is outside that subset.

This notebook includes the original JPEG and expert mask. You do not need to download
the whole dataset. The trained models are **not** embedded in the submitted notebooks:
you must make the actual `.keras` checkpoints available.

The image is an **illustrative comparison**, not proof that one pipeline is superior.
Do not switch images after seeing outputs merely to favour Attention U-Net.
The notebooks used different data splits, ROI processing, and classification thresholds.
The file labelled U-Net++ implements a standard U-Net. This figure labels it accordingly.
If your teammate has a different, genuine U-Net++ checkpoint/notebook, use that version
and its own preprocessing instead of relabelling this standard U-Net.


## Before running: collect the trained models

Keep the frozen Attention U-Net run directory, including `models/` and `tables/`.
The default path below is the final candidate path in your completed reporting notebook.

For the other three pipelines, obtain **both** the segmenter and final classifier from
the teammate who trained them. Copy their model files to your Drive or change the paths
in Section 2 to where you saved them. Their default folder names are supplied below.

For the file labelled U-Net++, the checkpoints were saved to temporary Colab storage.
If that teammate's original trained runtime is still open, paste this cell there to
preserve its actual selected models. Do not run the original notebook from the top
solely to export them:

```python
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
comparison_export = Path('/content/drive/MyDrive/Melanoma_Comparison_Models')
comparison_export.mkdir(parents=True, exist_ok=True)
unet.save(comparison_export / 'unet_segmenter.keras')
cls_model.save(comparison_export / 'unet_classifier.keras')
print(comparison_export)
```

If the runtime has restarted, locate the saved `.keras` files instead. A notebook's
saved charts and text outputs do not contain its trained weights.

ResUNet saves `resunet_best.keras` and `efficientnet_b0_final.keras` in its Drive
`models` folder. TransUNet saves `TransUNet_ISIC2016.keras` and
`EfficientNetB0_Augmented_Final.keras` in the Drive `43022` folder. Use these final
models, rather than a different earlier classifier checkpoint.


## 1. Imports and connect Google Drive


In [ ]:
import base64
import gc
import hashlib
import json
import sys
import zipfile
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import tensorflow as tf
import cv2
from sklearn.model_selection import train_test_split
from google.colab import drive

drive.mount('/content/drive')
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


## 2. Saved model locations

Update these paths if your files are elsewhere. The ResUNet and TransUNet defaults
refer to their original authors' Drive folders; they need to exist in **your connected
Drive** for this combined run. You can copy the shared files there or point to your copies.

This cell checks all required files before running any model. It never substitutes
randomly initialised models or silently retrains a missing checkpoint.


In [ ]:
ATTENTION_RUN = Path('/content/drive/MyDrive/Melanoma_ISIC2016_Final_Candidate/20261007T163637_386595Z')
UNET_FOLDER = Path('/content/drive/MyDrive/Melanoma_Comparison_Models')
RES_FOLDER = Path('/content/drive/MyDrive/Melanoma_AI_ResUNet_Targeted_Malignant_Aug_Results/models')
TRANS_FOLDER = Path('/content/drive/MyDrive/43022')

MODELS = {
    'Attention U-Net': {'run_dir': ATTENTION_RUN},
    'U-Net (uploaded notebook)': {
        'segmenter': UNET_FOLDER / 'unet_segmenter.keras',
        'classifier': UNET_FOLDER / 'unet_classifier.keras',
    },
    'ResUNet': {
        'segmenter': RES_FOLDER / 'resunet_best.keras',
        'classifier': RES_FOLDER / 'efficientnet_b0_final.keras',
    },
    'TransUNet': {
        'segmenter': TRANS_FOLDER / 'TransUNet_ISIC2016.keras',
        'classifier': TRANS_FOLDER / 'EfficientNetB0_Augmented_Final.keras',
    },
}

required = [
    ATTENTION_RUN / 'models/attention_unet_best.keras',
    ATTENTION_RUN / 'models/selected_final_classifier.keras',
    ATTENTION_RUN / 'tables/selected_configuration.json',
]
for name, spec in MODELS.items():
    if name != 'Attention U-Net':
        required.extend([spec['segmenter'], spec['classifier']])
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(
        'Copy the trained files into your connected Drive or correct Section 2 paths.\n'
        'Missing files:\n' + '\n'.join(missing)
    )

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT_DIR = Path('/content/drive/MyDrive/Melanoma_Shared_Image_Comparison') / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('All eight models and the Attention configuration are available.')
print('Outputs:', OUTPUT_DIR)


## 3. Restore the embedded image and independently check its label/split

The JPEG, mask, and official label CSVs below came from the official ISIC 2016 archive.
The original image bytes are retained. Model predictions were not used for selection.

TransUNet's test membership is reconstructed from its original code and dataset order.
This assumes the saved checkpoints correspond to those supplied notebooks/runs. Keep
the original split records when available. The official test set has already been
viewed during Attention U-Net development, so this is a follow-up illustration.


In [ ]:
#@title Restore the selected image and verify its label/split
EMBEDDED_FILES = {'ISIC_0000056.jpg': {'base64': '...', 'sha256': 'e8ec351ef99d5c6200666b253e58887dfaf0c4e2736882d72b49f5d9c3b4cf48'}, 'ISIC_0000056_Segmentation.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAABeAAAARpCAAAAACD4fh8AAAgO0lEQVR4nO3d23riuBKAUXt/ef9X9r6YzgEwxgbZqiqtdTHTnc7BgPRHCAPTBAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAkMfc+wCgjmWaJpOKOIxFaGP582fzihD+1/sAoIbl6V+gF4GHFpbNv0IX7krCh57F3OSiN2MQPrG1VDe76MwQhB2WJ1Pl9VaMKUY/Rh+88l3xx9myb6vdLKMTD7LCC8vjH5a7D+z8BnAtawvYdn9+++96fn+4zTO6MPBgQ6vFt4lGD8YdPNNya8VMowPDDlacsW3+O9meP2oLLRlhcO+8B0X/7uH/fgROYnjBj/NPd1l5ZNYc5DQGF/yn18mM5iCn+ep9ABCAM9UpyeqB4fWuu0nIWTyTldH17jucRuABihJ4gKIEnsHZoaEugWdwHuKkLoEHKErgAYoSeOjMowCcReChN4XnJAIP3Sk853AKAYMLFFezkcYMKcYWqO/TZELSlleTZEDBqg4nsWBgNLHrbkbSkAdZGUzsvkc/PHIReMYSPqDhD5BE3CFkKFny+XdiLpOJynuMG0aSpe/T9D03l7u/wwEGDQPJ1PdHJitH2YMHKErgAYoSeMaRe4cm++HTgcADFCXwAEUJPGRhj4aDBB7SUHiOEXiAogQeoCiBZxgFNjgKXASuJPCMQhwZjsBDHl6OhkMEHqAogQcoSuAhDTs0HCPwAEUJPEBRAg9Z2KHhIIEHKErgAYoSeEjCDg1HCTyDSP9KBfrOYQIPKeg7xwk8QFECzxjS79DAcQIPUJTAAxQl8AzBDg0jEnhGoO8MSeABihJ4gKIEngEU2KHxPCfeIPDUV6Dv8I6v3gdAZEuBlaO6M67005fT/JQx9SCp0vfUNwK9GDas+tvFTINkmeYyTb+V6VYgCqNmRMuL2/2ukAcGSd89nZpl/2aucphBM5zvCj6/6R86uXOUPO7pLJeOr9p9N1k5zpgZzeud9ZVOzv99fO1Llp9/ffz87Z/TWvW+m60cZsgM5bHCrz7p3u0X7Wzqh6Ps1Y7SgSNJz4TlCONlEMs0791b3xv4A039YJi93lE6eCzpmbPsZrCM4Hn+Vm7/F6383V8/dgxvjrTfn3LgMYMhmLq8ZpQMYCuAjwPgVS7nfZ+262e9tOdszTH7Pk1mLy8ZIgPYuybf9dmfODja1o7k/luMW/dpMn15xQip72UDv895eTwbprVDw+3Zkcw7PmcQpi/bjJDighVw93gLdtxBmb5sM0IqC1jJo8+ZYpP5yyavJllWzEbWeAUzyEHgi4qZ9z8qvBIxBOcNP2oK3/dp2jrIFIcfgF+RbBN4+tFxOJUtmpKylNM+DZzJ9CopS+D/8/3c2Pde52ZcJi+vGCOF/BYyWR9/Xwft+xIkuwBdmLu8ZIsml+d7Gst00bNRT7DM2Y4YUrAKSOT5OeRl+mgFf8CrybvzZeN2falQ5OR2y+P5u3UUyuH7r1U5oK3Zu+8aXP0OT75UKxJymmQay9O/qeGgNm74nWNiWR4/8dmXGmYJCXwW99NrefovDGOZppVCH/4ezT+TKNztSmGcqfXfgBzn8rZx/G1bnn/9sbeHITYr+AQ+XqRlM9rl/djDFXbwGlxW/7jj5xCcwMc31Kz6d7YnxzTbp3nxjdw0yQg8sSwi8o5GhXfdF2NTLT6Tjp0+exrBvOPrBCMXt1d8As9+83TuiPn7O0Q9wnMTxSfwHLBnHf7Z9/87JAUkNrdPfAJPJLe/QRQkNDdPAgpPXBISmbNoAIoS+ASskYB3aEcOdmmISkMCs4LPwSQiKouPwIQjDxOJmFQkLDdNIgpPXFISkVslF40nsmM98WzY07l6k1F4otsRlWX2bNhLuG6zUXji2+zK6jt6W82fwlWajsIT30ZYNgewILXlNMl0TAHi+30HkbX39eYqapGSKUMuv6HZHruC1JbrMyuNJ5O970WiSE25OtNSeCra/k0gWMe4vjLTeOp59Z5UmnWAKys1hWdEsrWXs2gAihJ4gKLc18ltY49mtoNDVbq1kxV8bs8H+rz5r8AABD65Zw3XduCr9wHwobWdGHUHJimo4k/j5/UPQxmytZcVfA3f63gjH/ghCLVZwlORbu3kQVYgGwuXnQQeoCiBr81dWRiYwAPp2KPZR+CBfBR+F4EHElL4PQQeoCiBL86jrDAugQcyskezg8ADFCXw1dmjgWGZ/iNwZ5aK1OslK/gRmAhUZOHyksADSSn8KwIPUJTAAxQl8EBW9mheEPgheJQVRiTwAEUJPEBRAj8GezQwIBN/HB6QohwB2+b6GYrGU4uAbXP9jEflqUPBNtmDH48pAYMQeICiBB7Iy4bjJoEvadkc9vZoYAzmej3fcd+4bS17qELCtljBl7M8/AHqMsy3fPU+AM6zWN7A0ASgmocFzcpNbNFDHRq2wRZNLduPrn5/0umHAZcxnDcIfCm7xroJAYOwB1/ecnMXVt1hHFbw9S1P/gwlGNTPeYCilCdDfd78V0hOxp6xgh/B8ue/UI6h/YxffaUY6IxJyNa5XmpReEalZStcKcUoPOOSs3v24IEiLG/uCTxAUQIPUJTAA1XYo7kj8MV4mAn4JvAARQk8UIY9mlsCX409Gkam8DcEHihE4f+y3ivHAGd0svbNCr4cgxv4j8ADFCXw9VjCMzjblN/EoCYjnJHp2j+uiOqknvHo2j+2aKoz1GFYAg8UY1XzTeABihL46uzBw7AEvjh9h3EJfG36zoAM+28CD1Sj8P8IPFCOwv9H4EszzBmUoT9NkxNGizPKGZi6uQpqE3jGNnrgbNEAZY2+xBF4oK7BCy/wlQ0+uGH0STD6FlVhYw9s+DVu5qzgq9J3WDfQ3Bj3V1txA41heOFv5pbHDxVmBV/DstwWXd/hx7L6xxEM8nustp8xO9/9HZim6XtqLA8fqW6MS1nVMk/TXc9ndYdH88O6Z4j2DXEha1Jy+MQI8bMHn5W+Ay8IfFL6Drwi8MCQRlgkCTwwpvvCFyz+CI8zVFRwKMLlfk8snms+AarYxRmFvkML88NkqpVEWzQZ6Ts0sRTfp6n162oUtcYgBFMni1bwAEUJfEIW8MAeAg9wo84SSuABbpUpvMDnU2bwQVRVJlmdh4vHUWXsQWgV4ljhMoxG4OEa6ftoiyYdfYeLpJ9sAg/wRPbCCzzAM8kLL/AAT+UufPoHEcaTe8BBPnkzaQWfjb7DxfJOury/moax/L2V8o40SCxrKLMe9yh+gj7f/hW4UtJSJj3s8h5LPq9/GLhCzlTmPOri1jP++OZiwGVStjLlQRcn4xBSvlw6iwZgl3xrL4EHKErgw8m3SgBiEniAfdKtvgQeoCiBByhK4AGKEniAogQeoCiBByhK4KNJdyIWDCPdaxV89T4A/lJ3oB0r+DCWRd+BltLd5ShK2iG8fLm0gg9B3yG8fH0XeIA9EvZd4AH2yHg/W+AB9khYeIEH2CVf4TNuKxWUb+DAmHIl0woeoCiBByhK4AF2y7WdKvAARQk8QFECD1CUwAMUJfAAuzkPHoAABB6gKIGPINeptTCuXDs02Q63JHmHJLIF0wq+O30HziHwAEUJfG8W8JBFth0age9N3yGLdH0XeIBd8vVd4AH2SNh3ge/MDg3kkLHvAt9ZykED48k5VQUe4JWcfRd4gKoEvrOkCwMgAYHvTeEhvKzTVOABihJ4gKIEHmBb1h0agQfYlrbvAg+wKW/fBR5gU+IXFBH43hIPHiA2gQcoSuABttiD5112aCC4xJNU4DtLvDgAghN4gKIEHqAoge/NHg1wEoEH2JT3UVaBB9iWtvACD1CUwHeUdlkApOAhvl7+1X1WeYgvaSmt4DtZ7v4P0JrA96HrwOkEvgt9h1SSTlmB7yHpYAFyEfgO9B24gsBfT9+BSwg8QFECD1CUwF/ODg1wDYEHKErgAYoSeICiBB6gKIEHeCnnyRECD/BaysILPEBRAg9QlMBfLeUdPSDj1BX4i2UcJMCUcvIK/LUSDhHgP/mmb9K3ks0p3/AAbuVKZq6jzU3foYBM0bRFA1CUwAMckOmuuMBfJtOwACoQeICiBB7giER3xgUeoCiBByhK4AEOybNHI/AARQn8ZTI9/w2oQOABjkmzRyPwAEUJ/HXs0QCXEniAogQeoCiBv5A9Gqghy6OsAg9wVJLCCzxAUQJ/JXs0wIUk52pJ7toBm1K00wr+aimGBfBCiqWawF9O4aGCDIUX+OspPFSQoPACD/CW+IUXeID3hC+8wAO8KXrhBR7gXcELL/AAb4tdeIHvwGk0UEbowmtND6GHBHDITUWXh4/0FOZAhiLwUFyMtMY4iuEoPJQXoK4BDmFMEg/V9c9r/yMYl8ZDbd376iyafrrf+EBtAt+RwgNnEvieFB4q674NK/BdKTxwHoHvS+GB0wg8QFECD1DUV+8DGNTyb3em+4MwQF32gDtQdRhC977aormevgOXEHiAouzBA5yh+waNFXwHdmhgBAH6LvAAJ4jQd4EHaC9E3wUeoLkYfRd4gNaC9F3gARqL0neBB2grTN8FHqAqgQcoSuCv5nlOUFucHRqBv5q+Q22B+i7wAFUJPEBRAn8tOzRQW6QdGoG/WKgbH2gt1hQXeIBWYvVd4AFaCdZ3gb9atAEA1CXwV1N44CICD9BGuOWbwAM0Ea7v01fvAwAoIF7dJyt4gAZC9l3gAT4Ws+8CD/CpoH0XeICqBB6gKIEHKErgAYoSeIDPRH2MVeABqhJ4gKIEHuAjYXdoAh9ZVd6VFSoJ3VAreIC3he67wAO8LXbfBf5ywQcEsF/06SzwAEUJPEBRAn+56HfqgJ3CT2aBv174QQHUIPAdKDxwBYHvQeGhgPgTWeAB3hG/7wLfR4KRAWzKMIszHGNNXpMGcktQTyv4XhIMDmBDgkWawAMUJfAARQk8wFvi79EIPEBRAt9L/F/+QHIC34uzaCC5+JNY4AGKEniAogQeoCiBByhK4AGKEniAogQeoCiBByhK4AGKEniAogS+F69FA5xM4DvRd+BsAt+HvkN28V9rTOABqhJ4gKIEvgs7NJBdgh0agQeoSuABihL4HuzQQHYZdmgEHqAqgQcoSuB7SHHnDnguxyT+6n0AAMnkqPtkBQ9wUJq+JzrSWpxHAzmlaqYVfB+pBgnwLdfUFXiAvXL1XeB7STZOgCnfvBX4XuZsQwXIRuD7UXjgVALfkcIDZxL4nhQeOJHAAxQl8F1ZwgPnEfi+nEsDnEbge1N44CQCD1CUwAMUJfDd2aMBziHwAEUJPEBRAt+fPRrgFAIfgMJDCummqsBHkG7YABlISxTepRWiS9dLK/go0g0dGE2+SSrwYeQbPDCUhFM04SFXZp8GosoYy4zHXJvGQ0wJa2mLJpqEgwiISeABihJ4gKIEHqAogQfYIePDYwIPUJTAAxSV8V5Hbc6Dh3iSltIKPpqkAwkqyzotBR5gW9a+CzzAtrR9F/hw7MFDLHn7LvDR6DvEkrjvAh+MvkNPjzXP3PfcB1+NukNP8zQ9TMPcifzqfQD80HfoZ73kufue/fBLEXjo5jeFy8rHssp/CeoQeOjjvoNLlTIWuRgV6Dv0UTeDzqIJo+4gg9AKTz2BB4ZWuO8CD1CVwMdReSEBdCAqsXikFa5UvIBW8LEUH24QS/UJJ/DAqKr3XeCBUZXvu8ADVCXwAEUJPEBRAg9QlMADY6r/GKvAA1Ql8LF4JivQjMADQxpgh0bgAaoS+FhGWFQAF/nqfQAAF5uXQZZSg1zMPDzKCucaKXojXdYcFB7OM1jx7MFHM9gAhCuNNr0EHhjFaH0XeGAYw22ACjxAUQIPUJTAA8MYbY9G4AGKEnhgGKOdRiPwAEUJPEBRXmwMGMNo+zOTwAP1DZj2/9iiAYobtu8CH85oJ+rC2cbtu8ADtQ3cd4EHShu572Nf+Jjs0UArowfOWTRANaN3/YctGqAYff8m8EAt+v5D4IFS9P2XwIdjeAJtCDxAUQIPUJTAx2OPBmhC4ANSeHib6fOHwEdkiAINCDxAUV6qACjDnd9bAg/UoO4PbNEAJej7I4GPyCsGw1H6vkLgA9J3OErf1wh8PPoOR+n7KoEH0tP3dQIPUJTAh2OHBmhD4AGKEniAogQ+Gjs0QCMCH4y+A60IfCz6DjQj8KHoO9COwEei7/AWU2edwAdikMKbTJ5VAh+HIQo0JfAARQk8UIA7wGsEPgwDFD5gAq0QeKAEhX8k8EANCv9A4MPwlgXwGYW/J/AARQk8QFECH4c9GviMPZo7Ag9QlMADFCXwQBW2Oe8IfCBGJ9CSwAMUJfBAEe4D3xP4SIxP+ICzJO9JSjTGKLxLz+64QmKSeThOz+7YoonJQIXjLIzuCDxAUQIPUJTAB2WPBviUwAMUJfAARQl8VPZogA/JSGTLNM1O/IL9FO2GqyM8hYfdFO2GLRqAogQeoCiBD899TuA96pGCfXjYRdFuuDqy0Hh4TdFuuDoS0XjYJmi3XB+5aDxsELRbHmTNxfgFdhP4ZBQenjI97gh8NoYwsJPAAxQl8EAR7t7eE/h0DGJYZWo8EHigBH1/JPAARQl8PhYqwC4CD1Rg4bNC4IEKvIzHCoEHKErggQps0awQeICiBD4hSxVgD63IyQNK8JeSrbKCz8lwhj9MiHUCD2Sn708IPJCcvj8j8EkZ0vCPyfCUwAOp6ftzAg9kpu8bBD4rwxomE2GbayczZ8MzNv16wQo+M8OboZkArwg8kJO+v/TV+wAAjlP3PVxLudmFZxRa9QYreCA8dX+PwAORafsHXHnJ2aOhMoH6jOsvO4WnHl1qxGmS2ZkKlGNQtyLwQCz63ozAA6HoezsCn57pQC0eVmpH4PNTeGpR+GYEvgCFB9YIfAUKD6wQeCAYezStCDxAUQIPUJTAA8F4UKkVgQcoSuABivJ68EAk9mcasoIHAtH3lgQeiEPfmxJ4IA7PcWpK4IFAFL4lgQciWSS+HYHPbLn7P8AfHtJIS9WpSpZasYLPSt+BFwQeoCiBB4Jx97QVgU/KFKAww7sRgQfCUfg2BB6IR+GbEPicDH/gJS8XnI62A/t4RkEWyzxN6s44tKkBV2IKus5wtKkBe/AZ6DvwBoFPQN+Bdwg8QFECDwRkC74Fp0nGtkzTbIcGeItfk4EpO+OSphZs0cSl74xL35twNQal7oxLllpxTcak74xKkxqyRROSvjMqfW9J4CPSd0al700JPBCGvrcl8EAU+t6YwAdkh4Yx6XtrAh+QYc6QDPzmBB4IQd/bE3ggAn0/gcBHZKgzHIP+DAIPUJTAAxQl8ABFCTxAUQIfkWc6MRqPsZ5C4AGKEniAogQ+IndXGYwhfw6BByhK4IHunFdwDoEHKErgAYoSeKA/ezSnEHiAogQ+IqsZoAGBj8hJwUADAg/0Z1FzCoEHKErgAYoSeKA7OzTnEPiInEXDWPT9JF+9DwAYm7qfR+CBftT9VLZoArJDwyj0/VwCD/Si7ycTeKATfT+bwMdjhwZoQuDD0XegDYEHKErggU7cWz2bwEdjzDMOo/1kAh+NEwsYiMKfS+CBfhT+VAIPUJTAAxQl8EBH9mjOJPAARQk8QFECD/Rkj+ZEAg9QlMBHYz0DNOIt+0JRd6AdgY9D3YGmbNGEoe9AWwIP9OTl9U4k8ABFCXwUdmiAxtw9CkHdGZUEnckKPgJ9B04g8AHoO3AGgQf6sUNzKoEHKErg+7NDA5xC4PtzJ5VhGfznEniAogQeoCiBByhK4IFunGFwLoHvzxgHTiHwQDfOojmXwAMUJfDd2aEBziHwQC92aE4m8EAn+n42gQf60PfTCTzQhb6fT+B78xgrcBKB70zfgbMIfF/6zqjs0FxA4AGKEnigB/deLyDwXRnjjMvoP5/AA30o/Ok80NGVEc7wNOhErtyuBB5k6Dy2aACKEnjgTLP1eT8CD5xo3rEDY6vyLAIPnGf++Q8dCDxwGmnvS+C7MvwpzQDvTOCBk+ztu98DZxF4gKIEvi9LF0bwYpw7i+YsX70PAChqvvmjindgBQ9cYWMV747sWQS+M0ObURjr1xN44BQPQVf4ywk8cJEnhRf+0wh8bwY341gd7abAeQS+O8ObkvbG3AQ4kcD3Z4BT0N7tGMP/TAIPXEjQr+SJTsCV/j7nSe1PJvDAxXT9KrZoAIoSeICiBB6gKIEH2rPNHoLAAxQl8P15nWzgFALfnzuzlGNQxyDwAEUJPEBRAh+Au7MUY0gHIfARmA6UYkBHIfAhmBBAewIfg8IDzQl8EApPGQZzGAIfhUlBQrNxG5qbJw7PaCWbefo7budpmhZJicStEYjCk8t/+Vhu/kYkbpNIFJ5UfvKxKElMbpZQFJ5M5CM6D7KGYsKQiOEansDHYsoAzQg88B47iuEJPEBRAh+MPRqgFYGPRuFJwx5NdAIPUJTAAxQl8OHYoyENezTBCTxAUQIfjyU80ITAB+Q1toEWpCQqu5skICCxWcFHZeYAHxL4sBQe+IyKhGafhuAUJDQ3T3QaT2gSEpktmujMH0KzAolMPjIwhwhMROKygs/ADALeIPApKDxwnMDnoPCEZQcxLoEHPqPwYQk88CGFj0rgk7BHQ2AKH5TAZ6HwwEGykYl1EkEJSUxW8JmYRQRl7RGTwAMUJfAARQk8QFECD1CUwGfikSyiMjZDEvhEzCHiWm7+RwxOvMvD1CG0+XuIqkoYboo09J0kVCUMWzRZ6DtwkMAnoe+kYbCGIfAARQk8QFECn4M7vcBhX70PgNfUHXiHFXx8+g68ReABihJ4gKIEPjw7NMB7BD48z/smGUM2DIEHKErgAYoS+Pjc4SUVAzYOgU/AhCERwzUQgc/AlCENgzUSt0YWzpYkA0UJxc2RiMYTnaDE4vbIReMJTVBisQefi/kD7CbwySg8sJfAZ6PwwE5ykZGdeIISlFis4DMyi4AdBB6gKIEHKErgAYoSeKAZj//HIvBAOwofisADFCXwAEUJPNCQPZpIBB6gKIEHKErgU/JaBcBrAg80ZPERicADFCXwOVkmAS8JPNCOpUcoAg9Q1FfvAwDKsH4PRuCBJtQ9HrdJVp4RTiRKEpI9eOBj+h6TLZqi5mmyyucq+h6UFXxW21Nq3vE50IiBFpUVfAnzzWrddBva+s3v3tyQrOArmKe/01rfO5lDXPNPDmKOcXRcy42e18+abP7zkXn9U2hpntau2vWPXm3vjD56pPPGV6hIWG6a1B6K/vjvNPd9lS+PH+1+he+f0McOdeu3l4jE5baprHtuSvo7Z+4f+bjiGp/vf/Ldv+z1+li/f9L6b7Q3fyxXcttUJvAneJwyy+0u2WU///5nHZ3Mr4519ZLu+jyicNtUJvDtvZoxZ1/n9z9/efovr20f696zcTQkMDdOYfp+gpcz5txrfe3HL08+/tLmoT77jn9/oywCEpzbpzKFb+/1jDn1Wm86Yd8K/M9XaUcCnugEo5qfFn4r3vNk5Z6GwEMWzas6/93fObAyl/cs3FKl2aNpbseMaXmtzx89inqUlXk1XqoAGmsYyfnPd7ugvV7OoBpbNHDAtQWc//3Hypr3WMGXJgup/S7e3ZC8ReBrE4ZgjqTajcenjKHqPM7a0s75cvcSNbfnp+y+RUxOPmUMlafw7Rx/Ld7f1+l6/hoyn/40eMYYqk/h99mxvj4c+I/eXcnk5FP24OvzXj67zDf/a/K9nr67UoufAa8ZaqOwjt+2awflyHRZtj55161hcvIpY2gYCr/lybt4bHzWp/a+3wa8zxgah8I/t++tbFvPlndejx0OsAc/DsF4at7861U/FloT+IHoyV4KTw1G2FDs0qx79j5Jrz6nhav2gxiRUTSajxo/V/kd8fj80r6u+m3CYAyk4XxQ6PnDrw9j7fmlfd1eq1GOiuzswbNpfvhzgfgEfJXG+elf4H2G0niOLMF/1+zHX0olrJCD/u71yaAFb/gxnu83kHgZ6hdPuM+q2MWB52zRDGmep9ehKxrCohcLVgj8wLZTVzSEUS9W1OMiNYEf2YBRSXCRExwiWQg8QFECz7rn60grzFO4WmnPqBrb0zNpPn4x85iCj/c4T7yiBuNpcE9ivTku8gbecGcsRvzoVmv9YljkLLyxznAMetZ6/XJc5Gu8oc54jHqm6b1XM8zVeCOdATmLhml6L3+pkpnqYKER455pmt58udrIa/jfVwQ2yhmWoc80Te++HnncwhvYYIuGf94L4vzwgupBwhrkMKAvE4EfP+vxQ6PiZhUf5D39DGuYJjOBW8sbQ+L+rSoCFN6whmkyE2jgfuXfv/CGNUyTmUAb9yv/zo03rGGazARO0rfwhjVMk7NoAMoSeFJ4OB9zU/9HASAC92U5Q9vC/h2l+76zcQ2TicAJWq+fbwfpru9uXMNki4b2zu27dsNuAk9w5V/oEk4j8MSm1fA2gaex8x5fff4hYI3JQmNNA/9sfK7/kHmapsWYhm8mA421DPzz4bn2UwxmuGWLhrg2im3rBl4TeBpr19lj30nf4d5X7wNgDEHeCQSGYgXPFebp8IL8xavPWLDDS6YJzd0v1edn/7Bu55i8+2ZGMjwwLTjB/fu0rn78ib1D8u/3MoxhjZnBOZ69g/erxu8fkd/fyRiGJ0wOTvKvvw8jbLvwxwakZzXBFvOD8yyrw2tt8/zZeh/4gLNoOM+eWs+7PxM4yMTick8eHrXfAo2ZUvTwX+ONPgAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAABO9X8+ihmLBwwJAgAAAABJRU5ErkJggg==', 'sha256': '65b158f463e90c415a0109b85e4fca6ca32c630bc68e2fbebe34ab0adf342771'}, 'ISBI2016_ISIC_Part3B_Training_GroundTruth.csv': {'base64': '...', 'sha256': '0ffadd63b7cec650cddfbe5df54d1d3167453fe6004c2bbf05824184b0c4d405'}, 'ISBI2016_ISIC_Part3B_Test_GroundTruth.csv': {'base64': 'SVNJQ18wMDAwMDAzLDAuMA0KSVNJQ18wMDAwMDEyLDAuMA0KSVNJQ18wMDAwMDEzLDEuMA0KSVNJQ18wMDAwMDE0LDAuMA0KSVNJQ18wMDAwMDE1LDAuMA0KSVNJQ18wMDAwMDIwLDAuMA0KSVNJQ18wMDAwMDIyLDEuMA0KSVNJQ18wMDAwMDIzLDAuMA0KSVNJQ18wMDAwMDI3LDAuMA0KSVNJQ18wMDAwMDM2LDEuMA0KSVNJQ18wMDAwMDM3LDAuMA0KSVNJQ18wMDAwMDQwLDEuMA0KSVNJQ18wMDAwMDQzLDEuMA0KSVNJQ18wMDAwMDUyLDAuMA0KSVNJQ18wMDAwMDUzLDAuMA0KSVNJQ18wMDAwMDU2LDEuMA0KSVNJQ18wMDAwMDU3LDAuMA0KSVNJQ18wMDAwMDY0LDAuMA0KSVNJQ18wMDAwMDY2LDAuMA0KSVNJQ18wMDAwMDY5LDAuMA0KSVNJQ18wMDAwMDcxLDAuMA0KSVNJQ18wMDAwMDcyLDAuMA0KSVNJQ18wMDAwMDg4LDAuMA0KSVNJQ18wMDAwMDkyLDAuMA0KSVNJQ18wMDAwMDk4LDAuMA0KSVNJQ18wMDAwMDk5LDAuMA0KSVNJQ18wMDAwMTAxLDAuMA0KSVNJQ18wMDAwMTA3LDAuMA0KSVNJQ18wMDAwMTExLDAuMA0KSVNJQ18wMDAwMTEzLDAuMA0KSVNJQ18wMDAwMTE1LDAuMA0KSVNJQ18wMDAwMTE3LDAuMA0KSVNJQ18wMDAwMTI1LDAuMA0KSVNJQ18wMDAwMTI2LDAuMA0KSVNJQ18wMDAwMTI5LDAuMA0KSVNJQ18wMDAwMTMyLDAuMA0KSVNJQ18wMDAwMTM2LDAuMA0KSVNJQ18wMDAwMTM4LDAuMA0KSVNJQ18wMDAwMTQxLDEuMA0KSVNJQ18wMDAwMTQ5LDEuMA0KSVNJQ18wMDAwMTYwLDEuMA0KSVNJQ18wMDAwMTYxLDEuMA0KSVNJQ18wMDAwMTY0LDEuMA0KSVNJQ18wMDAwMTY1LDEuMA0KSVNJQ18wMDAwMTY5LDEuMA0KSVNJQ18wMDAwMTcyLDEuMA0KSVNJQ18wMDAwMTc0LDEuMA0KSVNJQ18wMDAwMTc3LDAuMA0KSVNJQ18wMDAwMTg4LDAuMA0KSVNJQ18wMDAwMTk1LDAuMA0KSVNJQ18wMDAwMTk2LDAuMA0KSVNJQ18wMDAwMTk3LDAuMA0KSVNJQ18wMDAwMTk4LDAuMA0KSVNJQ18wMDAwMjAwLDAuMA0KSVNJQ18wMDAwMjAxLDAuMA0KSVNJQ18wMDAwMjAyLDAuMA0KSVNJQ18wMDAwMjEyLDAuMA0KSVNJQ18wMDAwMjEzLDAuMA0KSVNJQ18wMDAwMjIyLDAuMA0KSVNJQ18wMDAwMjI2LDAuMA0KSVNJQ18wMDAwMjI3LDAuMA0KSVNJQ18wMDAwMjI4LDAuMA0KSVNJQ18wMDAwMjMwLDAuMA0KSVNJQ18wMDAwMjMxLDAuMA0KSVNJQ18wMDAwMjMzLDAuMA0KSVNJQ18wMDAwMjM0LDAuMA0KSVNJQ18wMDAwMjQ2LDAuMA0KSVNJQ18wMDAwMjUzLDAuMA0KSVNJQ18wMDAwMjU0LDAuMA0KSVNJQ18wMDAwMjU3LDAuMA0KSVNJQ18wMDAwMjU4LDAuMA0KSVNJQ18wMDAwMjcxLDAuMA0KSVNJQ18wMDAwMjc5LDEuMA0KSVNJQ18wMDAwMjg3LDEuMA0KSVNJQ18wMDAwMjg5LDEuMA0KSVNJQ18wMDAwMjk4LDEuMA0KSVNJQ18wMDAwMjk5LDEuMA0KSVNJQ18wMDAwMzAyLDEuMA0KSVNJQ18wMDAwMzA2LDEuMA0KSVNJQ18wMDAwMzEwLDEuMA0KSVNJQ18wMDAwMzExLDEuMA0KSVNJQ18wMDAwMzE5LDAuMA0KSVNJQ18wMDAwMzIwLDAuMA0KSVNJQ18wMDAwMzI1LDAuMA0KSVNJQ18wMDAwMzI3LDAuMA0KSVNJQ18wMDAwMzI4LDAuMA0KSVNJQ18wMDAwMzQwLDAuMA0KSVNJQ18wMDAwMzQzLDAuMA0KSVNJQ18wMDAwMzU0LDAuMA0KSVNJQ18wMDAwMzU3LDAuMA0KSVNJQ18wMDAwMzY4LDAuMA0KSVNJQ18wMDAwMzcxLDAuMA0KSVNJQ18wMDAwMzc3LDAuMA0KSVNJQ18wMDAwMzc4LDAuMA0KSVNJQ18wMDAwMzgwLDAuMA0KSVNJQ18wMDAwMzg4LDAuMA0KSVNJQ18wMDAwMzkyLDAuMA0KSVNJQ18wMDAwMzkzLDEuMA0KSVNJQ18wMDAwNDA3LDAuMA0KSVNJQ18wMDAwNDE4LDAuMA0KSVNJQ18wMDAwNDIwLDEuMA0KSVNJQ18wMDAwNDMzLDAuMA0KSVNJQ18wMDAwNDQ4LDAuMA0KSVNJQ18wMDAwNDQ5LDAuMA0KSVNJQ18wMDAwNDU2LDAuMA0KSVNJQ18wMDAwNDY2LDEuMA0KSVNJQ18wMDAwNDcwLDAuMA0KSVNJQ18wMDAwNDc2LDAuMA0KSVNJQ18wMDAwNDc5LDAuMA0KSVNJQ18wMDAwNDgyLDEuMA0KSVNJQ18wMDAwNDg0LDEuMA0KSVNJQ18wMDAwNDg3LDEuMA0KSVNJQ18wMDAwNDkwLDAuMA0KSVNJQ18wMDAwNDk0LDAuMA0KSVNJQ18wMDAwNDk5LDAuMA0KSVNJQ18wMDAwNTAxLDAuMA0KSVNJQ18wMDAwNTA4LDAuMA0KSVNJQ18wMDAwNTA5LDAuMA0KSVNJQ18wMDAwNTEwLDAuMA0KSVNJQ18wMDAwNTE1LDAuMA0KSVNJQ18wMDAwNTE4LDEuMA0KSVNJQ18wMDAwNTI1LDAuMA0KSVNJQ18wMDAwNTI3LDAuMA0KSVNJQ18wMDAwNTMzLDEuMA0KSVNJQ18wMDAwNTM0LDAuMA0KSVNJQ18wMDAwNTM3LDAuMA0KSVNJQ18wMDAwNTM5LDAuMA0KSVNJQ18wMDAwNTQwLDAuMA0KSVNJQ18wMDAwNTQ3LDEuMA0KSVNJQ18wMDAwNTQ5LDEuMA0KSVNJQ18wMDAwNTUwLDEuMA0KSVNJQ18wMDAxMTAwLDEuMA0KSVNJQ18wMDAxMTAzLDEuMA0KSVNJQ18wMDAxMTI4LDEuMA0KSVNJQ18wMDAxMTMxLDEuMA0KSVNJQ18wMDAxMTQyLDEuMA0KSVNJQ18wMDAxMTgxLDAuMA0KSVNJQ18wMDAxMTg1LDAuMA0KSVNJQ18wMDAxMTg2LDAuMA0KSVNJQ18wMDAxMTkwLDAuMA0KSVNJQ18wMDAxMjA0LDAuMA0KSVNJQ18wMDAxMjQyLDAuMA0KSVNJQ18wMDAxMjk5LDAuMA0KSVNJQ18wMDAxNDI3LDAuMA0KSVNJQ18wMDAxNDg0LDAuMA0KSVNJQ18wMDAxNjg1LDAuMA0KSVNJQ18wMDAxOTYwLDAuMA0KSVNJQ18wMDAyMTA3LDAuMA0KSVNJQ18wMDAyMjQ2LDAuMA0KSVNJQ18wMDAyNjczLDAuMA0KSVNJQ18wMDAyODI5LDAuMA0KSVNJQ18wMDAyODcxLDAuMA0KSVNJQ18wMDAzMDU2LDAuMA0KSVNJQ18wMDAzNTU5LDAuMA0KSVNJQ18wMDAzNzI4LDAuMA0KSVNJQ18wMDA0MTE1LDAuMA0KSVNJQ18wMDA0MzQ2LDAuMA0KSVNJQ18wMDA1MDAwLDAuMA0KSVNJQ18wMDA2MTkzLDAuMA0KSVNJQ18wMDA2NzExLDAuMA0KSVNJQ18wMDA3MzIyLDAuMA0KSVNJQ18wMDA3NjkzLDAuMA0KSVNJQ18wMDA4MTE2LDAuMA0KSVNJQ18wMDA4MjA3LDAuMA0KSVNJQ18wMDA4NDA2LDAuMA0KSVNJQ18wMDA4NTA3LDAuMA0KSVNJQ18wMDA4NjAwLDAuMA0KSVNJQ18wMDA4NjI2LDAuMA0KSVNJQ18wMDA4NjU5LDAuMA0KSVNJQ18wMDA4OTkyLDAuMA0KSVNJQ18wMDA4OTk4LDAuMA0KSVNJQ18wMDA5MDM1LDAuMA0KSVNJQ18wMDA5MDc4LDAuMA0KSVNJQ18wMDA5MDgzLDAuMA0KSVNJQ18wMDA5MjAxLDAuMA0KSVNJQ18wMDA5Mjk4LDAuMA0KSVNJQ18wMDA5NTY0LDAuMA0KSVNJQ18wMDA5ODY5LDAuMA0KSVNJQ18wMDA5ODcyLDAuMA0KSVNJQ18wMDA5ODc0LDAuMA0KSVNJQ18wMDA5ODc5LDAuMA0KSVNJQ18wMDA5ODgwLDAuMA0KSVNJQ18wMDA5ODgxLDAuMA0KSVNJQ18wMDA5ODgyLDEuMA0KSVNJQ18wMDA5ODg1LDAuMA0KSVNJQ18wMDA5ODg5LDAuMA0KSVNJQ18wMDA5ODkxLDAuMA0KSVNJQ18wMDA5ODk4LDEuMA0KSVNJQ18wMDA5OTAxLDAuMA0KSVNJQ18wMDA5OTAyLDAuMA0KSVNJQ18wMDA5OTA2LDAuMA0KSVNJQ18wMDA5OTE4LDEuMA0KSVNJQ18wMDA5OTIwLDAuMA0KSVNJQ18wMDA5OTIzLDAuMA0KSVNJQ18wMDA5OTI2LDAuMA0KSVNJQ18wMDA5OTI3LDEuMA0KSVNJQ18wMDA5OTI4LDAuMA0KSVNJQ18wMDA5OTMwLDAuMA0KSVNJQ18wMDA5OTMxLDAuMA0KSVNJQ18wMDA5OTQzLDAuMA0KSVNJQ18wMDA5OTQ1LDAuMA0KSVNJQ18wMDA5OTQ2LDAuMA0KSVNJQ18wMDA5OTQ4LDAuMA0KSVNJQ18wMDA5OTU0LDAuMA0KSVNJQ18wMDA5OTU1LDEuMA0KSVNJQ18wMDA5OTU2LDAuMA0KSVNJQ18wMDA5OTU4LDAuMA0KSVNJQ18wMDA5OTU5LDEuMA0KSVNJQ18wMDA5OTY1LDAuMA0KSVNJQ18wMDA5OTcwLDAuMA0KSVNJQ18wMDA5OTc3LDAuMA0KSVNJQ18wMDA5OTgwLDAuMA0KSVNJQ18wMDA5OTgyLDAuMA0KSVNJQ18wMDA5OTg4LDAuMA0KSVNJQ18wMDA5OTkwLDAuMA0KSVNJQ18wMDA5OTkyLDEuMA0KSVNJQ18wMDA5OTkzLDEuMA0KSVNJQ18wMDA5OTk0LDAuMA0KSVNJQ18wMDA5OTk4LDAuMA0KSVNJQ18wMDEwMDA0LDAuMA0KSVNJQ18wMDEwMDA5LDAuMA0KSVNJQ18wMDEwMDExLDEuMA0KSVNJQ18wMDEwMDEzLDAuMA0KSVNJQ18wMDEwMDE2LDEuMA0KSVNJQ18wMDEwMDE4LDAuMA0KSVNJQ18wMDEwMDIwLDAuMA0KSVNJQ18wMDEwMDIzLDEuMA0KSVNJQ18wMDEwMDI4LDAuMA0KSVNJQ18wMDEwMDMzLDAuMA0KSVNJQ18wMDEwMDM0LDEuMA0KSVNJQ18wMDEwMDM3LDAuMA0KSVNJQ18wMDEwMDM4LDAuMA0KSVNJQ18wMDEwMDQxLDAuMA0KSVNJQ18wMDEwMDQ3LDAuMA0KSVNJQ18wMDEwMDU1LDAuMA0KSVNJQ18wMDEwMDU4LDAuMA0KSVNJQ18wMDEwMDU5LDAuMA0KSVNJQ18wMDEwMDYxLDAuMA0KSVNJQ18wMDEwMDYyLDAuMA0KSVNJQ18wMDEwMDczLDAuMA0KSVNJQ18wMDEwMDc3LDAuMA0KSVNJQ18wMDEwMDg4LDAuMA0KSVNJQ18wMDEwMDg5LDEuMA0KSVNJQ18wMDEwMDkyLDAuMA0KSVNJQ18wMDEwMTAwLDAuMA0KSVNJQ18wMDEwMTAzLDAuMA0KSVNJQ18wMDEwMTcxLDEuMA0KSVNJQ18wMDEwMTczLDAuMA0KSVNJQ18wMDEwMTc1LDEuMA0KSVNJQ18wMDEwMTgwLDAuMA0KSVNJQ18wMDEwMTgzLDAuMA0KSVNJQ18wMDEwMTg3LDAuMA0KSVNJQ18wMDEwMTkwLDAuMA0KSVNJQ18wMDEwMTkyLDAuMA0KSVNJQ18wMDEwMTkzLDAuMA0KSVNJQ18wMDEwMjAxLDAuMA0KSVNJQ18wMDEwMjAyLDAuMA0KSVNJQ18wMDEwMjA2LDAuMA0KSVNJQ18wMDEwMjA3LDEuMA0KSVNJQ18wMDEwMjE1LDAuMA0KSVNJQ18wMDEwMjE2LDAuMA0KSVNJQ18wMDEwMjI5LDAuMA0KSVNJQ18wMDEwMjMxLDEuMA0KSVNJQ18wMDEwMjM0LDAuMA0KSVNJQ18wMDEwMjM4LDAuMA0KSVNJQ18wMDEwMjU0LDAuMA0KSVNJQ18wMDEwMjU1LDAuMA0KSVNJQ18wMDEwMjU3LDAuMA0KSVNJQ18wMDEwMjYxLDAuMA0KSVNJQ18wMDEwMzI2LDAuMA0KSVNJQ18wMDEwMzMxLDAuMA0KSVNJQ18wMDEwMzM2LDAuMA0KSVNJQ18wMDEwMzQwLDAuMA0KSVNJQ18wMDEwMzQ2LDAuMA0KSVNJQ18wMDEwMzQ3LDAuMA0KSVNJQ18wMDEwMzQ4LDAuMA0KSVNJQ18wMDEwMzYwLDEuMA0KSVNJQ18wMDEwMzY4LDAuMA0KSVNJQ18wMDEwMzY5LDEuMA0KSVNJQ18wMDEwMzczLDAuMA0KSVNJQ18wMDEwMzc0LDAuMA0KSVNJQ18wMDEwMzc3LDAuMA0KSVNJQ18wMDEwMzc4LDAuMA0KSVNJQ18wMDEwMzc5LDEuMA0KSVNJQ18wMDEwNDM3LDAuMA0KSVNJQ18wMDEwNDQ0LDAuMA0KSVNJQ18wMDEwNDQ4LDAuMA0KSVNJQ18wMDEwNDQ5LDAuMA0KSVNJQ18wMDEwNDUxLDAuMA0KSVNJQ18wMDEwNDUyLDAuMA0KSVNJQ18wMDEwNDU0LDEuMA0KSVNJQ18wMDEwNDU2LDAuMA0KSVNJQ18wMDEwNDYwLDAuMA0KSVNJQ18wMDEwNDYzLDAuMA0KSVNJQ18wMDEwNDc0LDAuMA0KSVNJQ18wMDEwNDc3LDAuMA0KSVNJQ18wMDEwNDgzLDAuMA0KSVNJQ18wMDEwNDk0LDAuMA0KSVNJQ18wMDEwNDk4LDAuMA0KSVNJQ18wMDEwNTUyLDAuMA0KSVNJQ18wMDEwNTUzLDAuMA0KSVNJQ18wMDEwNTU2LDAuMA0KSVNJQ18wMDEwNTY1LDAuMA0KSVNJQ18wMDEwNTc0LDAuMA0KSVNJQ18wMDEwNTgyLDAuMA0KSVNJQ18wMDEwNTg0LDAuMA0KSVNJQ18wMDEwNTg3LDEuMA0KSVNJQ18wMDEwNTg4LDAuMA0KSVNJQ18wMDEwNTkxLDEuMA0KSVNJQ18wMDEwNTkyLDAuMA0KSVNJQ18wMDEwNTk2LDEuMA0KSVNJQ18wMDEwNTk3LDEuMA0KSVNJQ18wMDEwNTk4LDAuMA0KSVNJQ18wMDEwNTk5LDAuMA0KSVNJQ18wMDEwNjA0LDAuMA0KSVNJQ18wMDEwNjA2LDAuMA0KSVNJQ18wMDEwNjA3LDAuMA0KSVNJQ18wMDEwODQ2LDAuMA0KSVNJQ18wMDEwODQ3LDAuMA0KSVNJQ18wMDEwODU0LDAuMA0KSVNJQ18wMDEwODU2LDAuMA0KSVNJQ18wMDEwODYzLDEuMA0KSVNJQ18wMDExMDgwLDAuMA0KSVNJQ18wMDExMDgxLDAuMA0KSVNJQ18wMDExMDgzLDAuMA0KSVNJQ18wMDExMDg5LDAuMA0KSVNJQ18wMDExMDkwLDAuMA0KSVNJQ18wMDExMDkyLDAuMA0KSVNJQ18wMDExMDk4LDAuMA0KSVNJQ18wMDExMTAxLDAuMA0KSVNJQ18wMDExMTA0LDAuMA0KSVNJQ18wMDExMTA3LDAuMA0KSVNJQ18wMDExMTEwLDAuMA0KSVNJQ18wMDExMTEyLDEuMA0KSVNJQ18wMDExMTIyLDEuMA0KSVNJQ18wMDExMTI5LDAuMA0KSVNJQ18wMDExMTMyLDEuMA0KSVNJQ18wMDExMTQzLDAuMA0KSVNJQ18wMDExMTQ4LDAuMA0KSVNJQ18wMDExMTQ5LDAuMA0KSVNJQ18wMDExMTUwLDAuMA0KSVNJQ18wMDExMTUxLDEuMA0KSVNJQ18wMDExMTUyLDAuMA0KSVNJQ18wMDExMTU1LDAuMA0KSVNJQ18wMDExMTYyLDAuMA0KSVNJQ18wMDExMTY3LDAuMA0KSVNJQ18wMDExMTY4LDAuMA0KSVNJQ18wMDExMTcxLDAuMA0KSVNJQ18wMDExMTcyLDAuMA0KSVNJQ18wMDExMTc1LDAuMA0KSVNJQ18wMDExMTc2LDEuMA0KSVNJQ18wMDExMTc3LDAuMA0KSVNJQ18wMDExMjA0LDAuMA0KSVNJQ18wMDExMjA1LDAuMA0KSVNJQ18wMDExMjEzLDAuMA0KSVNJQ18wMDExMjE5LDAuMA0KSVNJQ18wMDExMjI0LDAuMA0KSVNJQ18wMDExMjI3LDAuMA0KSVNJQ18wMDExMjkyLDAuMA0KSVNJQ18wMDExMjk0LDAuMA0KSVNJQ18wMDExMjk4LDAuMA0KSVNJQ18wMDExMzAwLDAuMA0KSVNJQ18wMDExMzA1LDEuMA0KSVNJQ18wMDExMzEwLDAuMA0KSVNJQ18wMDExMzE5LDAuMA0KSVNJQ18wMDExMzIxLDAuMA0KSVNJQ18wMDExMzI1LDAuMA0KSVNJQ18wMDExMzMzLDAuMA0KSVNJQ18wMDExMzM2LDAuMA0KSVNJQ18wMDExMzM4LDAuMA0KSVNJQ18wMDExMzQ0LDEuMA0KSVNJQ18wMDExMzQ5LDEuMA0KSVNJQ18wMDExMzU5LDAuMA0KSVNJQ18wMDExMzYzLDAuMA0KSVNJQ18wMDExMzY3LDAuMA0KSVNJQ18wMDExMzc0LDEuMA0KSVNJQ18wMDExMzg0LDAuMA0KSVNJQ18wMDExMzg2LDAuMA0KSVNJQ18wMDExMzkyLDAuMA0K', 'sha256': '8f7718bfe5ed110e0235b0d9afc40fe89e52610c5687cc9cd7dd0c23352b82ce'}}

CASE_DIR = Path('/content/melanoma_shared_case')
CASE_DIR.mkdir(parents=True, exist_ok=True)
for name, item in EMBEDDED_FILES.items():
    data = base64.b64decode(item['base64'])
    if hashlib.sha256(data).hexdigest() != item['sha256']:
        raise ValueError('Embedded file checksum mismatch: ' + name)
    (CASE_DIR / name).write_bytes(data)

training = pd.read_csv(CASE_DIR / 'ISBI2016_ISIC_Part3B_Training_GroundTruth.csv',
                       header=None, names=['image_id', 'diagnosis'])
official_test = pd.read_csv(CASE_DIR / 'ISBI2016_ISIC_Part3B_Test_GroundTruth.csv',
                            header=None, names=['image_id', 'diagnosis'])
training['label'] = training['diagnosis'].str.strip().str.lower().map({'benign': 0, 'malignant': 1})
official_test['label'] = official_test['diagnosis'].astype(float).astype(int)
assert len(training) == 900 and len(official_test) == 379
assert training['label'].notna().all() and official_test['label'].isin([0, 1]).all()

combined = pd.concat([training[['image_id', 'label']],
                      official_test[['image_id', 'label']]], ignore_index=True)
_, trans_classifier_test = train_test_split(
    combined, test_size=0.20, stratify=combined['label'], random_state=42)
segmentation_order = pd.concat([
    training.sort_values('image_id')[['image_id', 'label']],
    official_test.sort_values('image_id')[['image_id', 'label']],
], ignore_index=True)
_, trans_segmenter_test = train_test_split(
    segmentation_order, test_size=0.20, stratify=segmentation_order['label'], random_state=42)
assert len(trans_classifier_test) == 256 and int(trans_classifier_test['label'].sum()) == 50

candidates = official_test.loc[
    (official_test['label'] == 1)
    & official_test['image_id'].isin(trans_classifier_test['image_id'])
    & official_test['image_id'].isin(trans_segmenter_test['image_id'])
].sort_values('image_id')
IMAGE_ID = str(candidates.iloc[0]['image_id'])
assert IMAGE_ID == 'ISIC_0000056', 'Split reconstruction changed. Recheck the source notebooks.'
assert IMAGE_ID not in set(training['image_id'])
IMAGE_PATH = CASE_DIR / f'{IMAGE_ID}.jpg'
MASK_PATH = CASE_DIR / f'{IMAGE_ID}_Segmentation.png'

split_audit = pd.DataFrame([
    {'Pipeline': 'Attention U-Net', 'Original train': 720, 'Original validation': 180,
     'Original test': 379, 'Original test melanomas': 75},
    {'Pipeline': 'U-Net (uploaded notebook)', 'Original train': 765, 'Original validation': 135,
     'Original test': 379, 'Original test melanomas': 75},
    {'Pipeline': 'ResUNet', 'Original train': 160, 'Original validation': 40,
     'Original test': 100, 'Original test melanomas': 19},
    {'Pipeline': 'TransUNet', 'Original train': 895, 'Original validation': 128,
     'Original test': 256, 'Original test melanomas': 50},
])
split_audit.to_csv(OUTPUT_DIR / 'original_evaluation_split_audit.csv', index=False)
display(split_audit)
print('Locked image:', IMAGE_ID, '| True label: Melanoma')
print('Eligible melanoma cases under the fixed rule:', len(candidates))
print('Reconstructed as held out from both TransUNet stages; outside other development sets.')
with Image.open(IMAGE_PATH) as image:
    display(image.copy())


## 4. Exact inference helpers

Attention U-Net uses the previously checked software module, including its saved
segmentation flip averaging, RGB ROI handling, quality-95 JPEG encoding, and locked
classification threshold. The other adapters reproduce the supplied notebooks.

All masks are scored against the same nearest-neighbour-resized 256 x 256 expert mask
using **hard per-image Dice and IoU**. These single-image scores are distinct from the
reported dataset averages, some of which used soft or batch-aggregated Dice.


In [ ]:
#@title Load the original preprocessing helpers
MODULE_DIR = Path("/content/melanoma_comparison_modules")
MODULE_DIR.mkdir(parents=True, exist_ok=True)
(MODULE_DIR / "attention_pipeline_core.py").write_text('"""Inference and Gradio UI for the frozen Attention U-Net melanoma candidate.\n\nNo training or threshold selection occurs in this module. JPEG ROI encoding is\npreserved because the training notebook classified saved quality-95 JPEG ROIs.\n"""\nfrom __future__ import annotations\n\nfrom io import BytesIO\nfrom pathlib import Path\nimport hashlib\nimport json\nimport threading\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nfrom scipy import ndimage\nimport tensorflow as tf\n\nSEG_SIZE = (256, 256)\nCLS_SIZE = (224, 224)\n\n\ndef image_tensor(path, size, normalise=False):\n    pixels = tf.io.decode_image(tf.io.read_file(str(path)), channels=3,\n                                expand_animations=False)\n    pixels = tf.cast(tf.image.resize(pixels, size, method="bilinear"), tf.float32)\n    return pixels / 255.0 if normalise else pixels\n\n\ndef roi_tensor(roi):\n    # Match roi.save(..., quality=95) -> decode_classifier_image() in training.\n    buffer = BytesIO()\n    roi.convert("RGB").save(buffer, format="JPEG", quality=95)\n    pixels = tf.io.decode_jpeg(buffer.getvalue(), channels=3)\n    return tf.cast(tf.image.resize(pixels, CLS_SIZE), tf.float32)\n\n\ndef crop_from_prediction(image_pil, predicted_mask, padding=0.15):\n    image = image_pil.convert("RGB")\n    width, height = image.size\n    binary = Image.fromarray(np.uint8(np.asarray(predicted_mask) > 0) * 255)\n    binary = np.asarray(binary.resize(image.size, Image.Resampling.NEAREST)) > 0\n    labelled, count = ndimage.label(binary)\n    reason = "cropped"\n    if count == 0:\n        crop, reason = image, "empty predicted mask"\n    else:\n        sizes = np.bincount(labelled.ravel())\n        sizes[0] = 0\n        lesion = labelled == int(sizes.argmax())\n        area = int(lesion.sum())\n        if area < max(64, int(width * height * 0.001)) or area > width * height * 0.80:\n            crop, reason = image, "implausible predicted area"\n        else:\n            ys, xs = np.where(lesion)\n            span = max(np.ptp(xs) + 1, np.ptp(ys) + 1)\n            margin = int(np.ceil(span * padding))\n            x0 = max(0, int(xs.min()) - margin)\n            x1 = min(width, int(xs.max()) + margin + 1)\n            y0 = max(0, int(ys.min()) - margin)\n            y1 = min(height, int(ys.max()) + margin + 1)\n            crop = image.crop((x0, y0, x1, y1))\n    a = np.asarray(crop)\n    h, w = a.shape[:2]\n    side = max(h, w)\n    top, left = (side - h) // 2, (side - w) // 2\n    square = np.pad(a, ((top, side - h - top), (left, side - w - left), (0, 0)),\n                    mode="edge")\n    roi = Image.fromarray(square).resize((CLS_SIZE[1], CLS_SIZE[0]),\n                                         Image.Resampling.BILINEAR)\n    return roi, reason\n\n\ndef mask_overlay(image, mask):\n    base = np.asarray(image.convert("RGB"), dtype=np.float32)\n    mask = Image.fromarray(np.uint8(mask) * 255).resize(\n        image.size, Image.Resampling.NEAREST)\n    binary = np.asarray(mask) > 127\n    tint = np.array([35, 195, 145], dtype=np.float32)\n    base[binary] = 0.65 * base[binary] + 0.35 * tint\n    boundary = binary & ~ndimage.binary_erosion(binary)\n    base[boundary] = [255, 215, 30]\n    return Image.fromarray(np.uint8(np.clip(base, 0, 255)))\n\n\ndef gradcam_overlay(roi, heatmap):\n    from matplotlib import colormaps\n    image = roi.convert("RGB").resize((CLS_SIZE[1], CLS_SIZE[0]))\n    heat = Image.fromarray(np.uint8(np.clip(heatmap, 0, 1) * 255)).resize(\n        image.size, Image.Resampling.BILINEAR)\n    colors = colormaps["jet"](np.asarray(heat, dtype=np.float32) / 255.0)[..., :3]\n    base = np.asarray(image, dtype=np.float32) / 255.0\n    return Image.fromarray(np.uint8(255 * np.clip(0.62 * base + 0.38 * colors, 0, 1)))\n\n\nclass MelanomaPipeline:\n    def __init__(self, segmenter, classifier, configuration, run_dir=None):\n        self.segmenter = segmenter\n        self.classifier = classifier\n        self.configuration = dict(configuration)\n        self.run_dir = Path(run_dir) if run_dir else None\n        self.lock = threading.RLock()\n        cfg = self.configuration\n        if cfg.get("view") != "dual":\n            raise ValueError("This app requires the full-image + ROI classifier (view=\'dual\').")\n        for field in ("threshold", "segmentation_threshold"):\n            if field not in cfg or not 0 <= float(cfg[field]) <= 1:\n                raise ValueError(f"Missing or invalid saved configuration field: {field}")\n        for field in ("method", "segmentation_method"):\n            if cfg.get(field) not in ("single", "4-flip average"):\n                raise ValueError(f"Unsupported inference method: {cfg.get(field)}")\n        if segmenter.input_shape[1:] != (*SEG_SIZE, 3):\n            raise ValueError("Segmenter input must be 256 x 256 RGB.")\n        if len(classifier.inputs) != 2:\n            raise ValueError("Expected two classifier inputs: full image and ROI.")\n        shapes = [tuple(x.shape[1:]) for x in classifier.inputs]\n        if shapes != [(*CLS_SIZE, 3), (*CLS_SIZE, 3)]:\n            raise ValueError(f"Classifier input shapes are incompatible: {shapes}")\n\n    @classmethod\n    def load(cls, run_dir):\n        run_dir = Path(run_dir)\n        paths = {\n            "configuration": run_dir / "tables" / "selected_configuration.json",\n            "segmenter": run_dir / "models" / "attention_unet_best.keras",\n            "classifier": run_dir / "models" / "selected_final_classifier.keras",\n        }\n        missing = [str(p) for p in paths.values() if not p.is_file()]\n        if missing:\n            raise FileNotFoundError("The saved pipeline is incomplete. Missing:\\n" + "\\n".join(missing))\n        configuration = json.loads(paths["configuration"].read_text())\n        provenance = run_dir / "tables" / "segmentation_configuration.json"\n        if provenance.is_file():\n            expected = json.loads(provenance.read_text()).get("checkpoint_sha256")\n            if expected:\n                actual = hashlib.sha256(paths["segmenter"].read_bytes()).hexdigest()\n                if actual != expected:\n                    raise ValueError("The segmentation checkpoint does not match its saved configuration.")\n        manifest_path = run_dir / "deployment_manifest.json"\n        if manifest_path.is_file():\n            for relative, expected in json.loads(manifest_path.read_text()).get("sha256", {}).items():\n                actual = hashlib.sha256((run_dir / relative).read_bytes()).hexdigest()\n                if actual != expected:\n                    raise ValueError(f"Deployment bundle file has changed: {relative}")\n        # compile=False avoids needing custom training losses for inference.\n        segmenter = tf.keras.models.load_model(paths["segmenter"], compile=False)\n        classifier = tf.keras.models.load_model(paths["classifier"], compile=False)\n        return cls(segmenter, classifier, configuration, run_dir)\n\n    def segmentation_probabilities(self, images, batch_size=4):\n        images = np.asarray(images, dtype=np.float32)\n        methods = [(), (2,), (1,), (1, 2)] if self.configuration[\n            "segmentation_method"] == "4-flip average" else [()]\n        predictions = []\n        for axes in methods:\n            transformed = np.flip(images, axis=axes) if axes else images\n            score = self.segmenter.predict(np.ascontiguousarray(transformed),\n                                            batch_size=batch_size, verbose=0)[..., 0]\n            score = np.flip(score, axis=axes) if axes else score\n            predictions.append(score)\n        return np.mean(np.stack(predictions), axis=0)\n\n    def classifier_probabilities(self, full_images, roi_images, batch_size=8):\n        full_images = np.asarray(full_images, dtype=np.float32)\n        roi_images = np.asarray(roi_images, dtype=np.float32)\n        methods = [(), (2,), (1,), (1, 2)] if self.configuration[\n            "method"] == "4-flip average" else [()]\n        predictions = []\n        for axes in methods:\n            full = np.flip(full_images, axis=axes) if axes else full_images\n            roi = np.flip(roi_images, axis=axes) if axes else roi_images\n            features = {"full_image": np.ascontiguousarray(full),\n                        "roi_image": np.ascontiguousarray(roi)}\n            predictions.append(self.classifier.predict(\n                features, batch_size=batch_size, verbose=0).ravel())\n        return np.mean(np.stack(predictions), axis=0)\n\n    def gradcam(self, full_batch, roi_batch, target_class):\n        model = self.classifier\n        backbone = model.get_layer("efficientnetb0")\n        gap = model.get_layer("gap")\n        with tf.GradientTape() as tape:\n            full_map = backbone(full_batch, training=False)\n            roi_map = backbone(roi_batch, training=False)\n            tape.watch(roi_map)\n            features = model.get_layer("combine_features")([gap(full_map), gap(roi_map)])\n            hidden = model.get_layer("hidden_dense")(features)\n            hidden = model.get_layer("head_dropout")(hidden, training=False)\n            output = model.get_layer("prediction")\n            logit = tf.linalg.matmul(hidden, output.kernel) + output.bias\n            score = logit[:, 0] if target_class == 1 else -logit[:, 0]\n        gradients = tape.gradient(score, roi_map)\n        if gradients is None:\n            raise RuntimeError("Grad-CAM could not trace the ROI feature map.")\n        weights = tf.reduce_mean(gradients, axis=(1, 2), keepdims=True)\n        heatmap = tf.nn.relu(tf.reduce_sum(roi_map * weights, axis=-1)[0])\n        return (heatmap / (tf.reduce_max(heatmap) + 1e-8)).numpy()\n\n    def predict_file(self, path, explain=True):\n        if path is None:\n            raise ValueError("Upload a dermoscopic lesion image first.")\n        path = Path(path)\n        with Image.open(path) as handle:\n            original = handle.convert("RGB")\n        with self.lock:\n            segmentation_input = image_tensor(path, SEG_SIZE, normalise=True)[None, ...]\n            probabilities = self.segmentation_probabilities(segmentation_input)[0]\n            mask = probabilities >= float(self.configuration["segmentation_threshold"])\n            roi, reason = crop_from_prediction(\n                original, mask, padding=float(self.configuration.get("roi_padding", 0.15)))\n            full_input = image_tensor(path, CLS_SIZE)[None, ...]\n            roi_input = roi_tensor(roi)[None, ...]\n            score = float(self.classifier_probabilities(full_input, roi_input)[0])\n            predicted = int(score >= float(self.configuration["threshold"]))\n            heatmap = self.gradcam(full_input, roi_input, predicted) if explain else None\n        result = {\n            "original": original, "mask": Image.fromarray(np.uint8(mask) * 255),\n            "mask_overlay": mask_overlay(original, mask), "roi": roi,\n            "score": score, "predicted_label": predicted,\n            "label": "Melanoma" if predicted else "Benign",\n            "roi_reason": reason, "probability_map": probabilities,\n        }\n        result["gradcam"] = gradcam_overlay(roi, heatmap) if heatmap is not None else None\n        return result\n\n\ndef build_demo(pipeline, sample_paths=()):\n    import gradio as gr\n    cfg = pipeline.configuration\n\n    def preview(path):\n        if path is None:\n            return None\n        with Image.open(path) as image:\n            return image.convert("RGB")\n\n    def analyse(path):\n        try:\n            result = pipeline.predict_file(path)\n        except (ValueError, FileNotFoundError, OSError) as error:\n            raise gr.Error(str(error)) from error\n        summary = f"### Model prediction: **{result[\'label\']}**\\n"\n        summary += (f"Melanoma model score: **{result[\'score\']:.4f}**. "\n                    f"Decision threshold: {float(cfg[\'threshold\']):.4f}.\\n\\n"\n                    "The score is a model output, not a calibrated clinical probability.")\n        if result["roi_reason"] != "cropped":\n            summary += "\\n\\nThe lesion crop was unreliable; the classifier used the full image for the ROI view."\n        panels = [(result["original"], "Original image"),\n                  (result["mask"], "Predicted lesion mask"),\n                  (result["mask_overlay"], "Lesion outline"),\n                  (result["roi"], "Lesion ROI"),\n                  (result["gradcam"], "Grad-CAM on ROI")]\n        return summary, panels\n\n    def clear():\n        return None, None, "Upload a dermoscopic lesion image, then select Analyse.", []\n\n    with gr.Blocks(title="Attention U-Net melanoma research demo",\n                   delete_cache=(3600, 3600)) as demo:\n        gr.Markdown("# Melanoma image analysis\\n"\n                    "Attention U-Net lesion segmentation with EfficientNetB0 classification.\\n\\n"\n                    "**Student research prototype. Not validated for clinical diagnosis.**")\n        with gr.Tab("Analyse image"):\n            with gr.Row():\n                with gr.Column(scale=1):\n                    upload = gr.File(label="Upload dermoscopic image (JPG or PNG)",\n                                     file_types=[".jpg", ".jpeg", ".png"], type="filepath")\n                    input_preview = gr.Image(label="Uploaded image", interactive=False)\n                    with gr.Row():\n                        analyse_button = gr.Button("Analyse", variant="primary")\n                        clear_button = gr.Button("Clear")\n                    if sample_paths:\n                        gr.Examples(examples=[[str(p)] for p in sample_paths],\n                                    inputs=upload, label="Public ISIC example images")\n                with gr.Column(scale=2):\n                    summary = gr.Markdown("Upload a dermoscopic lesion image, then select Analyse.")\n                    panels = gr.Gallery(label="Pipeline outputs", columns=3,\n                                        object_fit="contain", height="auto")\n                    gr.Markdown("The lesion mask estimates the lesion boundary. "\n                                "Grad-CAM shows image regions contributing to the model prediction; "\n                                "it is not a lesion mask.")\n            upload.change(preview, inputs=upload, outputs=input_preview)\n            analyse_button.click(analyse, inputs=upload, outputs=[summary, panels],\n                                 api_name="analyse", concurrency_limit=1)\n            clear_button.click(clear, outputs=[upload, input_preview, summary, panels])\n        with gr.Tab("Evaluation"):\n            gr.Markdown("## Saved evaluation results\\n"\n                        "Classification and segmentation measure different tasks. "\n                        "The official ISIC test set was viewed in earlier experiments; "\n                        "these results are exploratory follow-up.")\n            run_dir = pipeline.run_dir\n            displayed = False\n            if run_dir:\n                for filename, label in [("complete_segmentation_test_metrics.csv", "Lesion segmentation"),\n                                        ("complete_classification_test_metrics.csv", "Melanoma classification")]:\n                    path = run_dir / "tables" / filename\n                    if path.is_file():\n                        gr.Markdown("### " + label)\n                        gr.Dataframe(value=pd.read_csv(path).T.reset_index(),\n                                     headers=["Metric", "Value"], interactive=False)\n                        displayed = True\n                figures = sorted((run_dir / "figures").glob("*.png"))\n                if figures:\n                    gr.Gallery(value=[(str(p), p.stem.replace("_", " ")) for p in figures],\n                               label="Evaluation figures", columns=1, object_fit="contain")\n                    displayed = True\n            if not displayed:\n                gr.Markdown("Run the reporting notebook to add test loss, metrics and evaluation figures.")\n    return demo\n', encoding="utf-8")
(MODULE_DIR / "comparison_helpers.py").write_text('\nfrom pathlib import Path\nfrom io import BytesIO\nimport gc\nimport json\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport tensorflow as tf\nimport cv2\nfrom attention_pipeline_core import MelanomaPipeline\n\nIMG_SIZE = 256\nCLS_IMG_SIZE = 224\nMASK_MODE = "mask_crop"\nSEG_IMAGE_SIZE = (256, 256)\nSEG_THRESHOLD = 0.50\n\n\ndef get_lesion_region(image, unet_model, pad_frac=0.15, mode=MASK_MODE):\n    """Runs the trained U-Net on a full-res image to get a lesion mask, then applies\n    pixel-masking and/or cropping (per `mode`) before resizing for the classifier."""\n    h, w = image.shape[0], image.shape[1]\n    seg_input = tf.image.resize(image, [IMG_SIZE, IMG_SIZE]) / 255.0\n    seg_input = tf.expand_dims(seg_input, 0)\n    pred_mask_small = unet_model(seg_input, training=False)[0, ..., 0].numpy()\n    pred_mask_small = (pred_mask_small > 0.5).astype(np.uint8)\n\n    # Upsample the mask back to the original image resolution for pixel-accurate masking\n    pred_mask_full = tf.image.resize(\n        pred_mask_small[..., np.newaxis], [h, w], method="nearest"\n    ).numpy()[..., 0].astype(np.uint8)\n\n    ys, xs = np.where(pred_mask_small > 0)\n    lesion_found = len(xs) > 0 and len(ys) > 0\n\n    work_image = image.numpy() if hasattr(image, "numpy") else np.array(image)\n\n    if mode in ("mask", "mask_crop"):\n        if lesion_found:\n            work_image = work_image * pred_mask_full[..., np.newaxis]\n        # if no lesion detected, fall back to the unmasked image rather than an all-black frame\n\n    if mode in ("crop", "mask_crop") and lesion_found:\n        x_min, x_max = xs.min() / IMG_SIZE, xs.max() / IMG_SIZE\n        y_min, y_max = ys.min() / IMG_SIZE, ys.max() / IMG_SIZE\n        pad_x = (x_max - x_min) * pad_frac\n        pad_y = (y_max - y_min) * pad_frac\n        x_min = max(0.0, x_min - pad_x); x_max = min(1.0, x_max + pad_x)\n        y_min = max(0.0, y_min - pad_y); y_max = min(1.0, y_max + pad_y)\n\n        x1, x2 = int(x_min * w), int(x_max * w)\n        y1, y2 = int(y_min * h), int(y_max * h)\n        work_image = work_image[y1:max(y2, y1 + 1), x1:max(x2, x1 + 1), :]\n\n    out = tf.image.resize(work_image, [CLS_IMG_SIZE, CLS_IMG_SIZE])\n    return out.numpy()\n\ndef predict_mask_for_pil(model, image_pil):\n    resized = image_pil.resize(\n        (SEG_IMAGE_SIZE[1], SEG_IMAGE_SIZE[0]), Image.Resampling.BILINEAR\n    )\n    x = np.asarray(resized, dtype=np.float32) / 255.0\n    probability = model.predict(x[None, ...], verbose=0)[0, ..., 0]\n    binary = (probability >= SEG_THRESHOLD).astype(np.uint8)\n    return probability, binary\n\ndef extract_roi_from_binary_mask(image_pil, binary_mask, padding_fraction=0.10):\n    image_array = np.asarray(image_pil.convert(\'RGB\'))\n    h, w = image_array.shape[:2]\n    mask = np.asarray(\n        Image.fromarray((binary_mask * 255).astype(np.uint8)).resize(\n            (w, h), Image.Resampling.NEAREST\n        )\n    ) > 0\n\n    ys, xs = np.where(mask)\n    if len(xs) == 0 or len(ys) == 0:\n        return image_pil.copy(), (0, 0, w, h), True\n\n    x_min, x_max = int(xs.min()), int(xs.max())\n    y_min, y_max = int(ys.min()), int(ys.max())\n    box_w, box_h = max(1, x_max - x_min + 1), max(1, y_max - y_min + 1)\n    pad_x = int(round(box_w * padding_fraction))\n    pad_y = int(round(box_h * padding_fraction))\n    x0, y0 = max(0, x_min - pad_x), max(0, y_min - pad_y)\n    x1, y1 = min(w, x_max + pad_x + 1), min(h, y_max + pad_y + 1)\n\n    if (x1 - x0) < 8 or (y1 - y0) < 8:\n        return image_pil.copy(), (0, 0, w, h), True\n\n    return image_pil.crop((x0, y0, x1, y1)), (x0, y0, x1, y1), False\n\n\ndef hard_overlap(truth, prediction):\n    truth = np.asarray(truth, dtype=bool)\n    prediction = np.asarray(prediction, dtype=bool)\n    if truth.shape != prediction.shape:\n        raise ValueError(f"Mask shapes differ: {truth.shape} and {prediction.shape}")\n    intersection = int(np.logical_and(truth, prediction).sum())\n    total = int(truth.sum()) + int(prediction.sum())\n    union = int(np.logical_or(truth, prediction).sum())\n    return {\n        "dice": (2.0 * intersection / total) if total else 1.0,\n        "iou": (intersection / union) if union else 1.0,\n    }\n\n\ndef load_pair(segmenter_path, classifier_path):\n    seg = tf.keras.models.load_model(str(segmenter_path), compile=False)\n    cls = tf.keras.models.load_model(str(classifier_path), compile=False)\n    if tuple(seg.input_shape[1:]) != (256, 256, 3):\n        raise ValueError("Expected a 256 x 256 RGB segmenter.")\n    if len(cls.inputs) != 1 or tuple(cls.input_shape[1:]) != (224, 224, 3):\n        raise ValueError("Expected a single-input 224 x 224 RGB classifier.")\n    return seg, cls\n\n\ndef check_score(score):\n    score = float(score)\n    if not np.isfinite(score) or not 0 <= score <= 1:\n        raise ValueError(f"Invalid sigmoid model score: {score}")\n    return score\n\n\ndef infer_attention(image_path, run_dir):\n    pipeline = MelanomaPipeline.load(run_dir)\n    result = pipeline.predict_file(image_path, explain=False)\n    return {\n        "mask": np.asarray(result["mask"]) > 0,\n        "roi": result["roi"].copy(),\n        "score": check_score(result["score"]),\n        "threshold": float(pipeline.configuration["threshold"]),\n        "mask_threshold": float(pipeline.configuration["segmentation_threshold"]),\n        "mask_method": pipeline.configuration["segmentation_method"],\n        "classifier_method": pipeline.configuration["method"],\n        "roi_note": result["roi_reason"],\n    }\n\n\ndef infer_uploaded_unet(image_path, segmenter_path, classifier_path):\n    segmenter, classifier = load_pair(segmenter_path, classifier_path)\n    rgb = tf.cast(tf.io.decode_jpeg(tf.io.read_file(str(image_path)), channels=3),\n                  tf.float32)\n    seg_input = tf.image.resize(rgb, (256, 256)) / 255.0\n    probability = segmenter(seg_input[None, ...], training=False)[0, ..., 0].numpy()\n    # The supplied notebook uses > 0.5 for its ROI mask, then >= 0.5 for classification.\n    mask = probability > 0.5\n    roi_pixels = get_lesion_region(rgb, segmenter, pad_frac=0.15, mode="mask_crop")\n    score = check_score(classifier.predict(roi_pixels[None, ...], verbose=0).ravel()[0])\n    return {\n        "mask": mask,\n        "roi": Image.fromarray(np.clip(roi_pixels, 0, 255).astype(np.uint8)),\n        "score": score, "threshold": 0.5, "mask_threshold": 0.5,\n        "mask_method": "single", "classifier_method": "single",\n        "roi_note": "background masking + 15% bounding-box crop",\n    }\n\n\ndef infer_resunet(image_path, segmenter_path, classifier_path):\n    segmenter, classifier = load_pair(segmenter_path, classifier_path)\n    with Image.open(image_path) as image:\n        original = image.convert("RGB")\n    _, mask = predict_mask_for_pil(segmenter, original)\n    roi, bbox, fallback = extract_roi_from_binary_mask(original, mask, padding_fraction=0.10)\n    # Preserve the original saved quality-95 JPEG ROI preprocessing.\n    buffer = BytesIO()\n    roi.save(buffer, format="JPEG", quality=95)\n    roi_pixels = tf.io.decode_jpeg(buffer.getvalue(), channels=3)\n    roi_pixels = tf.cast(tf.image.resize(roi_pixels, (224, 224), method="bilinear"),\n                         tf.float32)\n    score = check_score(classifier.predict(roi_pixels[None, ...], verbose=0).ravel()[0])\n    return {\n        "mask": mask.astype(bool), "roi": roi.copy(),\n        "score": score, "threshold": 0.5, "mask_threshold": 0.5,\n        "mask_method": "single", "classifier_method": "single",\n        "roi_note": "full-image fallback" if fallback else "10% RGB bounding-box crop",\n    }\n\n\ndef infer_transunet(image_path, segmenter_path, classifier_path):\n    segmenter, classifier = load_pair(segmenter_path, classifier_path)\n    original = cv2.imread(str(image_path), cv2.IMREAD_COLOR)\n    if original is None:\n        raise ValueError("OpenCV could not decode the JPEG.")\n    original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)\n    # Match the supplied notebook\'s OpenCV resize and normalisation exactly.\n    seg_pixels = cv2.resize(original, (256, 256)).astype(np.float32) / 255.0\n    probability = segmenter.predict(seg_pixels[None, ...], verbose=0)[0, ..., 0]\n    mask = probability >= 0.5\n    masked_pixels = seg_pixels * mask[..., None].astype(np.float32)\n    roi_pixels = tf.image.resize(masked_pixels, (224, 224)).numpy() * 255.0\n    score = check_score(classifier.predict(roi_pixels[None, ...], verbose=0).ravel()[0])\n    return {\n        "mask": mask,\n        "roi": Image.fromarray(np.clip(roi_pixels, 0, 255).astype(np.uint8)),\n        "score": score, "threshold": 0.5, "mask_threshold": 0.5,\n        "mask_method": "single", "classifier_method": "single",\n        "roi_note": "256-pixel background-masked full frame, resized to 224",\n    }\n\n\ndef comparison_outputs(image_path, mask_path, output_dir, models):\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    with Image.open(mask_path) as image:\n        # One explicitly shared reference and hard Dice/IoU formula for this figure.\n        truth = np.asarray(image.convert("L").resize((256, 256),\n                           Image.Resampling.NEAREST)) > 127\n    outputs = []\n    rows = []\n    order = ["Attention U-Net", "U-Net (uploaded notebook)", "ResUNet", "TransUNet"]\n    for name in order:\n        tf.keras.backend.clear_session()\n        gc.collect()\n        spec = models[name]\n        print(f"Running {name} on {Path(image_path).name}...")\n        if name == "Attention U-Net":\n            result = infer_attention(image_path, spec["run_dir"])\n        else:\n            function = {\n                "U-Net (uploaded notebook)": infer_uploaded_unet,\n                "ResUNet": infer_resunet,\n                "TransUNet": infer_transunet,\n            }[name]\n            result = function(image_path, spec["segmenter"], spec["classifier"])\n        metrics = hard_overlap(truth, result["mask"])\n        label = "Melanoma" if result["score"] >= result["threshold"] else "Benign"\n        result.update(name=name, prediction=label, **metrics)\n        slug = {"Attention U-Net": "attention", "U-Net (uploaded notebook)": "unet",\n                "ResUNet": "resunet", "TransUNet": "transunet"}[name]\n        Image.fromarray(result["mask"].astype(np.uint8) * 255).save(output_dir / f"{slug}_mask.png")\n        result["roi"].save(output_dir / f"{slug}_roi.png")\n        outputs.append(result)\n        rows.append({\n            "Pipeline": name, "Image ID": Path(image_path).stem, "True label": "Melanoma",\n            "Prediction": label, "Model score": result["score"],\n            "Decision threshold": result["threshold"], "Hard Dice (this image)": metrics["dice"],\n            "Hard IoU (this image)": metrics["iou"], "Mask threshold": result["mask_threshold"],\n            "Mask inference": result["mask_method"],\n            "Classifier inference": result["classifier_method"], "ROI treatment": result["roi_note"],\n        })\n        tf.keras.backend.clear_session()\n        gc.collect()\n    table = pd.DataFrame(rows)\n    table.to_csv(output_dir / "single_image_comparison.csv", index=False)\n    return truth, outputs, table\n\n\ndef save_comparison_figures(image_path, truth, outputs, output_dir):\n    import matplotlib.pyplot as plt\n    output_dir = Path(output_dir)\n    with Image.open(image_path) as image:\n        original = image.convert("RGB").copy()\n    display_rgb = np.asarray(original.resize((256, 256), Image.Resampling.BILINEAR))\n    fig, axes = plt.subplots(1, 6, figsize=(19.5, 5.2), facecolor="white")\n    axes[0].imshow(display_rgb)\n    axes[0].set_title("Original image", fontsize=17)\n    axes[1].imshow(truth, cmap="gray", vmin=0, vmax=1)\n    axes[1].set_title("Expert mask", fontsize=17)\n    for ax, result in zip(axes[2:], outputs):\n        name = "U-Net*" if result["name"].startswith("U-Net (") else result["name"]\n        ax.imshow(result["mask"], cmap="gray", vmin=0, vmax=1)\n        ax.set_title(name, fontsize=17, fontweight="bold")\n        ax.text(0.5, -0.08,\n                f\'Dice {result["dice"]:.3f}\\n{result["prediction"]}\\n\'\n                f\'Score {result["score"]:.3f} / cutoff {result["threshold"]:.3f}\',\n                ha="center", va="top", transform=ax.transAxes, fontsize=13)\n    for ax in axes:\n        ax.axis("off")\n    fig.suptitle(f"One shared melanoma image: {Path(image_path).stem}", fontsize=23,\n                 fontweight="bold", y=0.97)\n    fig.subplots_adjust(left=0.015, right=0.985, top=0.80, bottom=0.27, wspace=0.12)\n    fig.text(0.5, 0.06,\n             "Illustrative case. Different development splits and pipeline settings. "\n             "*The uploaded U-Net++ notebook implements a standard U-Net.",\n             ha="center", fontsize=12)\n    fig.savefig(output_dir / "Slide1_Shared_Image_Comparison.png", dpi=200,\n                facecolor="white", bbox_inches="tight")\n    plt.show()\n\n    fig, axes = plt.subplots(4, 3, figsize=(10, 14), facecolor="white")\n    for i, result in enumerate(outputs):\n        axes[i, 0].imshow(result["mask"], cmap="gray", vmin=0, vmax=1)\n        axes[i, 1].imshow(display_rgb)\n        if result["mask"].any() and not result["mask"].all():\n            axes[i, 1].contour(result["mask"].astype(float), levels=[0.5],\n                               colors=["#00dd77"], linewidths=1.2)\n        axes[i, 2].imshow(result["roi"])\n        axes[i, 0].set_title(f\'{result["name"]}\\nDice {result["dice"]:.4f}\', fontsize=13)\n        axes[i, 1].set_title("Predicted boundary", fontsize=13)\n        axes[i, 2].set_title(f\'Classifier ROI\\nPrediction: {result["prediction"]}\', fontsize=13)\n    for ax in axes.flat:\n        ax.axis("off")\n    fig.suptitle(f"Pipeline details for {Path(image_path).stem} (true label: Melanoma)", fontsize=16)\n    fig.tight_layout(rect=(0, 0, 1, 0.97))\n    fig.savefig(output_dir / "Backup_Masks_Boundaries_and_ROIs.png", dpi=180,\n                facecolor="white", bbox_inches="tight")\n    plt.close(fig)\n', encoding="utf-8")
sys.path.insert(0, str(MODULE_DIR))
from comparison_helpers import comparison_outputs, save_comparison_figures


## 5. Run the same image through all four saved pipelines

Models load sequentially to limit GPU memory use. Each original decision threshold is
retained. Scores are raw model outputs, not interchangeable calibrated probabilities.


In [ ]:
truth, outputs, comparison_table = comparison_outputs(
    IMAGE_PATH, MASK_PATH, OUTPUT_DIR, MODELS)
display(comparison_table.round(4))
save_comparison_figures(IMAGE_PATH, truth, outputs, OUTPUT_DIR)

manifest = {
    'image_id': IMAGE_ID, 'true_label': 'Melanoma',
    'selection_rule': 'First melanoma ID in official test and reconstructed TransUNet test intersection',
    'source': 'https://isic-archive.s3.amazonaws.com/challenges/2016/ISBI2016_ISIC_Part3B_Test_Data.zip',
    'image_sha256': EMBEDDED_FILES[f'{IMAGE_ID}.jpg']['sha256'],
    'mask_sha256': EMBEDDED_FILES[f'{IMAGE_ID}_Segmentation.png']['sha256'],
    'reference_mask_resize': 'PIL nearest neighbour, 256 x 256',
    'metric': 'Per-image hard Dice and IoU, thresholded predicted masks',
    'models': {name: {key: str(value) for key, value in spec.items()}
               for name, spec in MODELS.items()},
    'limitations': [
        'Illustrative single case; not evidence of overall superiority',
        'Different development cohorts, preprocessing and thresholds',
        'Supplied U-Net++ notebook contains a standard U-Net',
        'Attention official test set previously viewed',
        'Held-out membership reconstructed from supplied code; verify checkpoint provenance',
    ],
}
(OUTPUT_DIR / 'comparison_provenance.json').write_text(json.dumps(manifest, indent=2))
print('Slide image:', OUTPUT_DIR / 'Slide1_Shared_Image_Comparison.png')


## 6. Download the figure and results

Use `Slide1_Shared_Image_Comparison.png` for the slide. The backup figure shows the
actual boundaries and classifier ROIs, which differ between these pipelines.
Do not calculate overall sensitivity, specificity, accuracy, or AUC from one image.


In [ ]:
from google.colab import files

package_path = OUTPUT_DIR / f'{IMAGE_ID}_Four_Pipeline_Comparison.zip'
with zipfile.ZipFile(package_path, 'w', compression=zipfile.ZIP_DEFLATED) as package:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file() and path != package_path:
            package.write(path, arcname=path.name)
    package.write(IMAGE_PATH, arcname=IMAGE_PATH.name)
    package.write(MASK_PATH, arcname=MASK_PATH.name)
files.download(str(package_path))


## What to say on the slide

"We used the same melanoma image to illustrate how each saved pipeline identifies
the lesion and classifies it. We selected Attention U-Net as our current prototype
because we prioritised melanoma sensitivity and it provided strong average lesion
segmentation. The original experiments used different test splits, so this is an
illustrative comparison. A matched evaluation is still needed to establish which
complete pipeline performs best."

Use this statement regardless of which model wins on this one image. If Attention
U-Net makes an error here, show it honestly or move the example to a backup slide.
Do not change thresholds for the selected image or replace its actual outputs.

For the results table, add a **test-set size** column, rename the supplied U-Net++
row to **U-Net** unless a correct alternate implementation is provided, and add
TransUNet's reported classification AUC **0.7787**. ResUNet used 100 test images,
TransUNet used 256, and the other two used 379. The current table does not establish
an architecture-only ranking.
